### 第 05 章 深度 Q 网络（DQN）


#### 本章概述

本章将介绍强化学习领域的一个重要里程碑——**深度 Q 网络（Deep Q-Network, DQN）**。DQN 是由 DeepMind 提出的一种将深度学习与强化学习成功结合的算法，在 Atari 游戏中达到了人类水平的表现。它成功克服了上一章提到的“致命三要素”带来的训练不稳定问题。

**学习目标**：
- 理解从 Q-Learning 到 DQN 的演进动机。
- 掌握 DQN 赖以成功的两大核心技术：经验回放（Experience Replay）与目标网络（Target Network）。
- 了解 DQN 的主流改进算法（Double DQN, Dueling DQN 等）。
- 学习 DQN 的实现细节和工程调试技巧。

**前置知识**：
- 需要掌握 Q-Learning 算法的更新公式。
- 掌握上一章中关于函数近似和神经网络在强化学习中应用的挑战。


##### 5.1 从 Q-Learning 到 DQN


**理论部分**

传统的 **Q-Learning** 是一种基于表格的方法。它通过维护一个 $Q$-table 来记录每一个状态-动作对 $(s, a)$ 的价值，其更新公式为：
$$ Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t) \right] $$

正如我们在第 4 章所讨论的，面对高维状态（如像素图像输入），表格型 Q-Learning 会遭遇**维度灾难**，且无法实现**泛化**。为此，我们引入了非线性函数近似——深度神经网络。

**DQN（Deep Q-Network）** 本质上就是**带有神经网络的 Q-Learning**。它使用一个参数为 $\theta$ 的神经网络 $Q(s, a; \theta)$ 来近似真实的 $Q$ 值。

按照 Q-Learning 的逻辑，其目标损失函数应该为均方误差：
$$ J(\theta) = \mathbb{E}_{(s, a, r, s')} \left[ \left( r + \gamma \max_{a'} Q(s', a'; \theta) - Q(s, a; \theta) \right)^2 \right] $$

然而，直接像上述这样用神经网络去拟合，往往会导致发散和训练崩溃。原因正是因为上一章提到的**致命三要素**：
1. **非静态目标**：我们用同一个网络来计算预测值 $Q(s, a; \theta)$ 和目标值 $r + \gamma \max_{a'} Q(s', a'; \theta)$。网络每次更新，目标也在跟着不断移动（狗咬尾巴问题）。
2. **样本相关性**：智能体在环境中的连续交互样本（状态序列）存在极强的时序相关性，打破了神经网络优化所需的独立同分布（i.i.d）假设。

为了解决这些问题，DQN 引入了以下将要介绍的两大核心技术。

**关键要点**：
- DQN 是深度学习与 Q-Learning 的结合。
- 简单地用神经网络替换 Q-table 无法工作，因为面临非静态目标和数据相关性导致的不稳定。


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# 我们首先定义 DQN 中的神经网络结构
# 这是一个简单的多层感知机 (MLP) 示例，用于向量输入
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
        
    def forward(self, x):
        # 输入状态 x，输出该状态下所有可选动作的 Q 值
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


##### 5.2 DQN 的核心技术


**理论部分**

**1. 经验回放（Experience Replay）**

经验回放的思路非常直接：我们不再使用智能体刚刚产生的连续转移来立刻更新网络。相反，我们将智能体与环境交互产生的数据元组 $e_t = (S_t, A_t, R_{t+1}, S_{t+1}, \text{done})$ 存储在一个大容量的**回放缓冲区（Replay Buffer）** $\mathcal{D}$ 中。

在训练时，我们从缓冲区中**随机采样（Random Sampling）**一个小批量（Mini-batch）的数据来更新网络参数。

**优点**：
- **打破相关性**：随机采样打乱了数据的时序顺序，极大降低了样本间的相关性，使得数据更符合 i.i.d 假设。
- **数据复用**：过去的经验可以被多次采样学习，提高了数据利用率。
- **平滑分布**：有助于稳定数据的分布，避免网络过拟合到当前这一小段轨迹。

**2. 目标网络（Target Network）**

为了解决目标不断移动带来的不稳定，DQN 引入了第二个网络：**目标网络 $\hat{Q}$**。它的结构与主网络 $Q$ 完全相同，参数分别为 $\theta^-$ 和 $\theta$。

- **主网络 $Q(s, a; \theta)$**：频繁更新。用于选择动作和评估当前 Q 值。
- **目标网络 $\hat{Q}(s, a; \theta^-)$**：参数被冻结，只在每隔 $C$ 步之后，才将主网络的参数复制给它（$\theta^- \leftarrow \theta$），或者采用软更新策略。

引入目标网络后，损失函数变为：
$$ J(\theta) = \mathbb{E}_{(s, a, r, s') \sim \mathcal{D}} \left[ \left( r + \gamma \max_{a'} \hat{Q}(s', a'; \theta^-) - Q(s, a; \theta) \right)^2 \right] $$
在这个公式中，目标项（含 $\hat{Q}$）因为参数冻结，在一段时间内变成了固定的标签，从而将强化学习在短期内转换为一个较为稳定的监督学习回归问题。

**关键要点**：
- **经验回放**：解决数据相关性，提高样本利用率。
- **目标网络**：提供静态的自举目标，稳定训练过程。


In [ ]:
from collections import deque

# 实现一个简单的经验回放缓冲区
class ReplayBuffer:
    def __init__(self, capacity):
        # 使用 deque 维持固定容量，超出容量时自动剔除最旧的数据
        self.buffer = deque(maxlen=capacity)
        
    def push(self, state, action, reward, next_state, done):
        # 将经验存入缓冲区
        self.buffer.append((state, action, reward, next_state, done))
        
    def sample(self, batch_size):
        # 随机采样一个批次的数据
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))
        # 转换为 Numpy 数组方便后续转换为 PyTorch Tensor
        return (np.array(state), np.array(action), np.array(reward, dtype=np.float32), 
                np.array(next_state), np.array(done, dtype=np.uint8))
    
    def __len__(self):
        return len(self.buffer)

# 演示目标网络的用法（伪代码逻辑片段）
state_dim = 4
action_dim = 2

# 1. 初始化主网络和目标网络
q_net = QNetwork(state_dim, action_dim)
target_net = QNetwork(state_dim, action_dim)

# 2. 将主网络的初始参数复制给目标网络
target_net.load_state_dict(q_net.state_dict())

# 3. 目标网络的参数在训练中不计算梯度
for param in target_net.parameters():
    param.requires_grad = False

# 4. 参数同步函数（每隔 C 步调用一次）
def sync_target_network():
    target_net.load_state_dict(q_net.state_dict())


##### 5.3 DQN 的改进算法


**理论部分**

基础的 DQN 仍存在一些缺陷，后续研究者提出了许多经典的改进变体。这些改进变体经常被组合使用。

**1. Double DQN (DDQN)**
- **问题**：标准 DQN 的目标计算中包含了 $\max_{a'} \hat{Q}(s', a'; \theta^-)$。由于使用同一个网络（或同样参数的旧网络）来同时**选择动作**和**评估该动作的价值**，往往会导致 Q 值被严重高估（Overestimation）。
>在标准的 DQN 中，我们用来更新神经网络参数 $\theta$ 的 TD 目标（TD Target）公式如下：
>$$\boxed{Y^\text{DQN}_t = r_{t+1} + \gamma \max_{a'} \hat{Q}(s', a')}$$
>这里的 $\max_{a'}$ 操作。DQN使用同一个目标网络 $\theta^-$ 来同时完成两件事：
>选择动作（找出那个让 Q 值最大的动作）。
>评估价值（用找到的最大 Q 值作为未来的期望收益）。
>
>DQN的TD target可以写作：$Y^\text{DQN}_t = r_{t+1} + \gamma \max_{a'} \hat{Q}(s', a) = r_{t+1} + Q(s', \argmax_a' \hat{Q}(s',a'))$. 换句话说，操作实际可以被拆解为两部分：首先选取状态下的最优动作: $a'^{*}=\argmax_{a'}Q_{\theta^-}(s',a')$，接着计算该动作对应的价值 $\hat{Q}(s', a'^*)$。 当这两部分采用同一套 $\hat{Q}$(目标网络)网络进行计算时，每次得到的都是神经网络当前估算的所有动作价值中的最大值。考虑到通过神经网络估算的值本身在某些时候会产生正向或负向的误差，在 DQN 的更新方式下神经网络会将正向误差累积。
>
>例如，我们考虑一个特殊情形：在状态 $s'$ 下所有动作的值 $Q$ 均为 0，即$Q(s',a'_i) = 0, \forall i$，此时正确的更新目标应为$r_{t+1} + 0$，但是由于神经网络拟合的误差通常会出现某些动作的估算有正误差的情况，即存在某个动作$a'$有 $Q(s',a') > 0$，此时我们的更新目标出现了过高估计，$r_{t+1} + \gamma \max_{a'} Q(s',a') > r_{t+1} + 0$。因此，当我们用 DQN 的更新公式进行更新时，$Q(s,a)$也就会被过高估计了。同理，我们拿这个$Q(s,a)$来作为更新目标来更新上一步的值时，同样会过高估计，这样的误差将会逐步累积。对于动作空间较大的任务，DQN 中的过高估计问题会非常严重，造成 DQN 无法有效工作的后果。

- **改进**：解耦动作的选择与价值的评估。使用当前的主网络 $Q$ 来选择未来状态 $s'$ 的最佳动作，用目标网络 $\hat{Q}$ 来评估这个动作的价值。
- **目标公式**：
  $$\boxed{Y^{\text{DDQN}}_t = r_{t+1} + \gamma \underbrace{\hat{Q} \left( s', \underbrace{\color{blue}{\argmax_{a'} Q(s',a')}}_{\text{通过策略网络选择动作}}\right)}_{\text{通过目标网络评估动作}}}$$
**2. Dueling DQN**
- **问题**：在很多状态下，无论采取什么动作对状态的价值影响都不大。标准的 DQN 只输出每个 $(s,a)$ 独立的 Q 值，没有利用这一特性。
- **改进**：修改网络架构，将 Q 值的输出拆分为两部分：**状态价值函数 $V(s)$** 和 **动作优势函数 $A(s, a)$**。最后在输出层再将两者相加。
  $$ Q(s, a) = V(s) + \left( A(s, a) - \frac{1}{|A|} \sum_{a'} A(s, a') \right) $$
- **优点**：即使不更新每个动作，网络也能学好状态本身 $V(s)$ 的价值，泛化能力更强。

**3. 优先级经验回放 (Prioritized Experience Replay, PER)**
- **问题**：标准的 Replay Buffer 采用均匀随机采样。但那些预测误差大（TD Error 较大）的经验，对网络学习更“有价值”，应该更频繁地被回放学习。
- **改进**：根据样本的 TD Error 大小为其赋予采样优先级。误差越大的样本被采样的概率越高。同时为了纠正因为有偏采样引入的偏差，需要引入重要性采样权重（Importance Sampling Weights）。

**4. Rainbow DQN**
- **总结性工作**：DeepMind 后来将 DQN 的所有重要改进（包括上述三个，加上多步自举 Multi-step Learning、分布强化学习 Distributional RL、噪声网络 Noisy Nets）集成在一起，形成了一个极其强大的算法组合，被称为 Rainbow。在相同的训练时间内，其性能远超基础版 DQN。

**关键要点**：
- Double DQN 解决 Q 值高估问题。
- Dueling DQN 从网络结构层面优化价值表示。
- PER 从数据采样层面提高学习效率。
- Rainbow 是 DQN 系列的集大成者。


In [ ]:
# 演示 Dueling DQN 网络架构的差异
class DuelingQNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super(DuelingQNetwork, self).__init__()
        # 提取共有特征的层
        self.feature_layer = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU()
        )
        
        # 价值流 (Value stream)：输出标量 V(s)
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        
        # 优势流 (Advantage stream)：输出向量 A(s, a)
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )
        
    def forward(self, x):
        features = self.feature_layer(x)
        
        V = self.value_stream(features)
        A = self.advantage_stream(features)
        
        # Q(s, a) = V(s) + (A(s, a) - mean(A(s, a')))
        # 注意这里需要利用 broadcasting 机制保证维度对齐
        Q = V + (A - A.mean(dim=1, keepdim=True))
        return Q


##### 5.4 DQN 的实现与调试技巧


**理论部分**

在实际手写和调试 DQN 算法时，初学者很容易遇到不收敛的情况。以下是一些极其重要的工程与调试技巧：

1. **探索策略（Exploration Strategy）**：
   - DQN 是离策略算法，收集数据通常使用 $\epsilon$-greedy 策略。
   - **技巧**：不要一开始就把 $\epsilon$ 设得太小。通常让 $\epsilon$ 从 1.0 开始，在训练的前期（例如总步数的前 10%~50%）逐渐退火（衰减）到 0.05 或 0.01。前期充分的随机探索对于填充高质量的 Replay Buffer 至关重要。

2. **回放缓冲区（Replay Buffer）预热**：
   - **技巧**：在正式开始网络更新之前，先使用纯随机策略运行环境一定步数，将 Replay Buffer 预填充一定量的数据（例如 10000 条）。如果 Buffer 里数据极少就开始训练，网络很容易过拟合到初期极度相关的少量状态上。

3. **平滑的目标网络更新**：
   - 虽然原始的 DQN 使用硬更新（Hard Update），即每隔 $C$ 步整体拷贝一次参数。
   - **技巧**：现代强化学习更多使用**软更新（Soft Update）**。即每一步都对目标网络进行微小的更新：$\theta^- \leftarrow \tau \theta + (1-\tau)\theta^-$，其中 $\tau$ 通常非常小（如 0.005）。这能提供更平滑的目标曲线。

4. **奖励塑形（Reward Shaping）与裁剪（Clipping）**：
   - 异常庞大或者波动巨大的奖励会导致梯度爆炸。
   - **技巧**：可以将回报裁剪到 $[-1, 1]$，或者在训练时裁剪 Huber Loss（PyTorch 中的 `SmoothL1Loss`）或裁剪梯度（`torch.nn.utils.clip_grad_norm_`），以防网络崩溃。

5. **日志与监控**：
   - 监控训练时的 Episode Return 是最基本的。
   - **技巧**：同时监控 **TD Error**（损失值）和 **平均预测的 Q 值**。如果平均 Q 值一直飙升发散，说明碰到了高估问题，需要检查目标网络更新频率或者切换到 Double DQN。


##### 5.5 实战：DQN、Double DQN 与 Dueling DQN 玩 CartPole 游戏


下面我们将用完整的代码，在经典的 `CartPole-v1`（倒立摆）环境中实现并对比标准 DQN、Double DQN 和 Dueling DQN 的效果。

我们把代码逻辑封装在一个类中，通过不同的标志位（`use_double` 和 `use_dueling`）来切换不同的变体。


In [ ]:
import gym
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque

# 1. 定义 Dueling DQN / Standard DQN 网络结构
class QNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128, use_dueling=False):
        super(QNet, self).__init__()
        self.use_dueling = use_dueling
        
        self.feature_layer = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU()
        )
        
        if self.use_dueling:
            self.value_stream = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1)
            )
            self.advantage_stream = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, action_dim)
            )
        else:
            self.q_layer = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, action_dim)
            )
            
    def forward(self, x):
        features = self.feature_layer(x)
        if self.use_dueling:
            V = self.value_stream(features)
            A = self.advantage_stream(features)
            Q = V + (A - A.mean(dim=1, keepdim=True))
        else:
            Q = self.q_layer(features)
        return Q

# 2. 定义经验回放缓冲区
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
        
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
        
    def sample(self, batch_size):
        states, actions, rewards, next_states, dones = zip(*random.sample(self.buffer, batch_size))
        return np.array(states), actions, rewards, np.array(next_states), dones
    
    def __len__(self):
        return len(self.buffer)

# 3. 定义 DQN 智能体
class DQNAgent:
    def __init__(self, state_dim, action_dim, use_double=False, use_dueling=False):
        self.action_dim = action_dim
        self.use_double = use_double
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 主网络和目标网络
        self.q_net = QNet(state_dim, action_dim, use_dueling=use_dueling).to(self.device)
        self.target_net = QNet(state_dim, action_dim, use_dueling=use_dueling).to(self.device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=1e-3)
        self.buffer = ReplayBuffer(10000)
        
        self.batch_size = 64
        self.gamma = 0.99
        
        # Epsilon-greedy 参数
        self.epsilon_start = 1.0
        self.epsilon_final = 0.01
        self.epsilon_decay = 500
        self.frame_idx = 0
        
    def select_action(self, state):
        epsilon = self.epsilon_final + (self.epsilon_start - self.epsilon_final) * \
                  math.exp(-1. * self.frame_idx / self.epsilon_decay)
        self.frame_idx += 1
        
        if random.random() > epsilon:
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            with torch.no_grad():
                q_values = self.q_net(state_tensor)
            action = q_values.max(1)[1].item()
        else:
            action = random.randrange(self.action_dim)
        return action
    
    def update(self):
        if len(self.buffer) < self.batch_size:
            return
        
        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)
        
        # 当前状态的 Q 值
        q_values = self.q_net(states)
        q_value = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)
        
        # 计算目标 Q 值
        with torch.no_grad():
            if self.use_double:
                # Double DQN：使用主网络选择动作，使用目标网络评估价值
                next_actions = self.q_net(next_states).max(1)[1]
                next_q_values = self.target_net(next_states)
                next_q_value = next_q_values.gather(1, next_actions.unsqueeze(1)).squeeze(1)
            else:
                # Standard DQN：直接使用目标网络选择并评估最大动作价值
                next_q_value = self.target_net(next_states).max(1)[0]
            
            expected_q_value = rewards + self.gamma * next_q_value * (1 - dones)
        
        loss = F.mse_loss(q_value, expected_q_value)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
    def sync_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

# 4. 训练循环函数
def train_agent(agent, env_name="CartPole-v1", max_episodes=200):
    # gym >= 0.26 之后 make api 略有改变，此处使用通用适配写法
    env = gym.make(env_name)
    rewards = []
    
    for episode in range(max_episodes):
        state = env.reset()
        # 兼容 Gym 新老版本返回格式 (老版本返回 state，新版本返回 (state, info))
        if isinstance(state, tuple):
            state = state[0]
            
        episode_reward = 0
        done = False
        truncated = False
        
        while not (done or truncated):
            action = agent.select_action(state)
            
            # 兼容 Gym 新老版本 step 返回格式
            step_result = env.step(action)
            if len(step_result) == 5:
                next_state, reward, done, truncated, _ = step_result
            else:
                next_state, reward, done, _ = step_result
                truncated = False
            
            agent.buffer.push(state, action, reward, next_state, done or truncated)
            agent.update()
            
            state = next_state
            episode_reward += reward
            
        # 每隔 10 个 episode 同步一次目标网络参数
        if episode % 10 == 0:
            agent.sync_target()
            
        rewards.append(episode_reward)
        if (episode + 1) % 50 == 0:
            print(f"Episode {episode+1}/{max_episodes}, Reward: {np.mean(rewards[-20:]):.2f}")
            
    env.close()
    return rewards

# ================= 实验对比 =================
state_dim = 4
action_dim = 2
max_episodes = 250

print("--- Training Standard DQN ---")
dqn_agent = DQNAgent(state_dim, action_dim, use_double=False, use_dueling=False)
dqn_rewards = train_agent(dqn_agent, max_episodes=max_episodes)

print("\n--- Training Double DQN ---")
ddqn_agent = DQNAgent(state_dim, action_dim, use_double=True, use_dueling=False)
ddqn_rewards = train_agent(ddqn_agent, max_episodes=max_episodes)

print("\n--- Training Dueling DQN ---")
dueling_agent = DQNAgent(state_dim, action_dim, use_double=False, use_dueling=True)
dueling_rewards = train_agent(dueling_agent, max_episodes=max_episodes)

# 绘制对比曲线
def moving_average(a, n=10):
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n

plt.figure(figsize=(10, 5))
plt.plot(moving_average(dqn_rewards), label="DQN", alpha=0.8)
plt.plot(moving_average(ddqn_rewards), label="Double DQN", alpha=0.8)
plt.plot(moving_average(dueling_rewards), label="Dueling DQN", alpha=0.8)
plt.title("Training Performance on CartPole-v1")
plt.xlabel("Episodes (Moving Average)")
plt.ylabel("Reward")
plt.legend()
plt.grid(True)
plt.show()


#### 本章小结

本章我们深入探讨了深度强化学习的奠基算法 DQN（Deep Q-Network）。它优雅地融合了 Q-Learning 与深度神经网络，并针对性地解决了两者结合带来的不稳定问题。

**关键概念回顾**：
- 深度 Q 网络（DQN）：使用神经网络近似 $Q(s, a)$。
- 经验回放（Experience Replay）：打破时序相关性，重复利用样本。
- 目标网络（Target Network）：提供稳定的更新目标标签。
- Double DQN：解耦选择动作和评估动作，防止 Q 值高估。
- Dueling DQN：将 Q 值的预估拆解为状态价值 V 和优势函数 A。
- $\epsilon$-greedy 衰减与软更新（Soft Update）。

**下一章预告**：
基于价值的方法（Value-based Methods）如 DQN 通常只适用于离散动作空间。如果动作是连续的（例如机器人的关节角度），找到 $\max_a Q(s, a)$ 会变得非常困难。下一章我们将学习**策略梯度方法（Policy Gradient Methods）**，直接优化策略本身，为连续控制打下基础。
